# Orbit Attitudes In NSTK

This notebook focuses on how `nstk.propagation.Orbit` configures and returns spacecraft attitudes.

Topics covered:
- the NSTK default attitude and why it is `vvlh`
- the difference between Orekit `vvlh` / `lvlh_ccsds` and `lvlh` / `qsw`
- common frame-axis conventions used with NSTK/Orekit
- all supported ways to configure an orbit attitude law
- how to apply fixed angular offsets with `LofOffset`
- how to inspect and compare returned quaternions

## Conventions

NSTK defaults orbit attitudes to `attitude="vvlh"`, implemented as `LofOffset(native_frame, LOFType.VVLH)`.

The local-orbital-frame names matter:
- `vvlh` and `lvlh_ccsds` are Orekit's CCSDS-style family.
- `lvlh` and `qsw` are Orekit's STK/Vallado-style LVLH family.
- `vvlh` is a good general Earth-observing default because it keeps the spacecraft in a nadir-pointing local orbital attitude law.
- Plain strings and `LOFType` values select an aligned local orbital frame. If you need a fixed angular bias from that frame, build an Orekit `LofOffset` provider explicitly.

For detailed API documentation, see `Orbit.set_attitude_law(...)`.

## Common axis conventions

The most common NSTK/Orekit attitude options use these conventions:

- `vvlh` / `lvlh_ccsds`: `+Z` opposite position, `+Y` opposite orbital momentum
- `lvlh` / `qsw`: `+X` along position, `+Z` along orbital momentum
- `tnw`: `+X` along velocity, `+Z` along orbital momentum
- `ntw`: `+Y` along velocity, `+Z` along orbital momentum
- `vnc`: `+X` along velocity, `+Y` along orbital momentum

For each LOF, the remaining axis is the one required to complete a right-handed triad.

Common Earth-pointing providers that are not pure LOF axis conventions:
- `nadir`: points toward the sub-satellite point on the WGS84 reference ellipsoid
- `body_center`: points directly toward the Earth's center

Those two are Orekit pointing laws, not just named LOF axis sets, so they are better thought of as geometric pointing behaviors rather than fixed axis definitions.

Two naming points matter in practice:
- In Orekit, `lvlh` and `qsw` are equivalent.
- In Orekit, `vvlh` is another name for the `lvlh_ccsds` axis family.

In [1]:
# Ensure local package import when running directly from examples/.
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "examples" else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [2]:
from astropy.time import Time
import numpy as np

from nstk.propagation import Orbit


def build_orbit(attitude=None):
    kwargs = {
        "epoch": Time("2026-01-01T00:00:00", scale="utc"),
        "a": 7050e3,
        "e": 0.002,
        "i": np.deg2rad(97.4),
        "raan": np.deg2rad(5.0),
        "argp": np.deg2rad(45.0),
        "anomaly": np.deg2rad(0.0),
        "anomaly_type": "mean",
    }
    if attitude is not None:
        kwargs["attitude"] = attitude
    return Orbit.from_kepler_two_body(**kwargs)


t_query = np.array([0.0, 60.0, 120.0], dtype=np.float64)

orbit_default = build_orbit()
orbit_vvlh = build_orbit("vvlh")

q_default = orbit_default.get_attitude(t_query)
q_vvlh = orbit_vvlh.get_attitude(t_query)

print("native frame:", orbit_default.get_native_frame().getName())
print("default equals explicit vvlh:", bool(np.allclose(q_default, q_vvlh)))
print("default quaternion[0] [q1, q2, q3, q4]:", q_default[0])

native frame: GCRF
default equals explicit vvlh: True
default quaternion[0] [q1, q2, q3, q4]: [ 0.064887   -0.91999912 -0.04290571  0.38412289]


## Quaternion convention used by NSTK

NSTK returns the attitude quaternion in STK-style scalar-last ordering `[q1, q2, q3, q4]`.

- `q1`, `q2`, `q3` are the vector terms
- `q4` is the scalar term
- the quaternion is the rotation from the attitude reference frame into the spacecraft/body frame
- in normal NSTK orbit usage, that reference frame is typically the orbit native frame
- Orekit itself exposes the same quaternion as scalar-first `[q0, q1, q2, q3]`; NSTK reorders it on output to match STK
- `q` and `-q` represent the same physical orientation

That last point matters when comparing quaternions numerically: a sign flip does not imply a different attitude.

In [3]:
from org.hipparchus.geometry.euclidean.threed import Vector3D  # type: ignore

state0 = orbit_default.propagator.propagate(orbit_default.propagator.getInitialState().getDate())
att0 = state0.getAttitude()
rot0 = att0.getRotation()

q1, q2, q3, q4 = q_default[0]
body_plus_z_in_reference = rot0.applyInverseTo(Vector3D.PLUS_K)

print("attitude reference frame:", att0.getReferenceFrame().getName())
print("quaternion ordering: [q1, q2, q3, q4] (scalar last)")
print("q1 q2 q3 vector:", q1, q2, q3)
print("q4 scalar:", q4)
print("unit norm:", float(np.linalg.norm(q_default[0])))
print(
    "body +Z axis expressed in the reference/native frame:",
    [
        float(body_plus_z_in_reference.getX()),
        float(body_plus_z_in_reference.getY()),
        float(body_plus_z_in_reference.getZ()),
    ],
)


attitude reference frame: GCRF
quaternion ordering: [q1, q2, q3, q4] (scalar last)
q1 q2 q3 vector: 0.06488700121375138 -0.9199991189602309 -0.042905710170443236
q4 scalar: 0.38412289468431243
unit norm: 0.9999999999999999
body +Z axis expressed in the reference/native frame: [-0.7123534950998548, 0.02909726564312025, -0.701217403628229]


## Comparing local orbital frame choices

This compares several built-in attitude choices. Equal quaternions usually indicate equivalent axis conventions for this orbit and time sample.

In [4]:
attitude_specs = [
    "vvlh",
    "lvlh_ccsds",
    "lvlh",
    "qsw",
    "tnw",
    "ntw",
    "vnc",
    "nadir",
    "body_center",
]

reference = q_default

for spec in attitude_specs:
    orbit = build_orbit(spec)
    quat = orbit.get_attitude(t_query)
    print(f"{spec:12s} matches default: {bool(np.allclose(reference, quat))}  first q: {quat[0]}")

vvlh         matches default: True  first q: [ 0.064887   -0.91999912 -0.04290571  0.38412289]
lvlh_ccsds   matches default: True  first q: [ 0.064887   -0.91999912 -0.04290571  0.38412289]
lvlh         matches default: False  first q: [-0.70595736  0.25694747 -0.27892876 -0.59816465]
qsw          matches default: False  first q: [-0.70595736  0.25694747 -0.27892876 -0.59816465]
tnw          matches default: False  first q: [ 0.31749794 -0.68087653  0.6201987   0.22573387]
ntw          matches default: False  first q: [-0.70595736  0.25694747 -0.27892876 -0.59816465]
vnc          matches default: False  first q: [-0.38412289  0.04290571 -0.91999912  0.064887  ]
nadir        matches default: False  first q: [-0.06492874  0.92058393  0.04274997 -0.38272958]
body_center  matches default: False  first q: [-0.064887    0.91999912  0.04290571 -0.38412289]


## Using Orekit `LOFType` values directly

You do not need to build a `LofOffset` manually when you only want one of Orekit's built-in local orbital frames.

In [5]:
from org.orekit.frames import LOFType  # type: ignore

orbit_loftype = build_orbit(LOFType.QSW)
q_loftype = orbit_loftype.get_attitude(t_query)

orbit_mapping = build_orbit({"type": "lof", "lof": LOFType.VVLH})
q_mapping = orbit_mapping.get_attitude(t_query)

print("LOFType.QSW first q:", q_loftype[0])
print("mapping with LOFType.VVLH matches default:", bool(np.allclose(q_mapping, q_default)))

LOFType.QSW first q: [-0.70595736  0.25694747 -0.27892876 -0.59816465]
mapping with LOFType.VVLH matches default: True


## Other supported attitude inputs

NSTK also accepts:
- a dict such as `{"type": "nadir"}`
- a callable returning an Orekit `AttitudeProvider`
- a prebuilt Orekit `AttitudeProvider`
- construction from an existing `SpacecraftState`

For built-in aligned local orbital frames, strings and `LOFType` values are enough. For fixed angular offsets from a LOF, use an explicit `LofOffset` provider.

In [6]:
from org.orekit.attitudes import LofOffset  # type: ignore
from org.orekit.frames import LOFType  # type: ignore


def qsw_callable(inertial_frame, iers, simple_eop):
    del iers, simple_eop
    return LofOffset(inertial_frame, LOFType.QSW)


orbit_custom = build_orbit()
orbit_custom.set_attitude_law({"type": "nadir"})
q_nadir = orbit_custom.get_attitude(t_query)

orbit_custom.set_attitude_law(qsw_callable)
q_callable = orbit_custom.get_attitude(t_query)

orbit_custom.set_attitude_law(LofOffset(orbit_custom.get_native_frame(), LOFType.QSW))
q_provider = orbit_custom.get_attitude(t_query)

state0 = orbit_default.propagator.getInitialState()
orbit_from_state = Orbit.from_spacecraft_state(state0, attitude="vvlh")

print("nadir first q:", q_nadir[0])
print("callable equals provider object:", bool(np.allclose(q_callable, q_provider)))
print("from_spacecraft_state first q:", orbit_from_state.get_attitude(0.0))

nadir first q: [-0.06492874  0.92058393  0.04274997 -0.38272958]
callable equals provider object: True
from_spacecraft_state first q: [ 0.064887   -0.91999912 -0.04290571  0.38412289]


## Angular offsets from a local orbital frame with `LofOffset`

Use Orekit `LofOffset` when you want a fixed angular bias from a local orbital frame rather than the aligned frame itself.

A practical pattern is to use `RotationOrder.ZYX`, where the three angles read naturally as:
- `alpha1`: yaw about LOF `Z`
- `alpha2`: pitch about LOF `Y`
- `alpha3`: roll about LOF `X`

The angles are interpreted with the Orekit/Hipparchus attitude convention used by `LofOffset`.

In [7]:
from org.hipparchus.geometry.euclidean.threed import RotationOrder  # type: ignore


yaw_10_deg_qsw = LofOffset(
    orbit_default.get_native_frame(),
    LOFType.QSW,
    RotationOrder.ZYX,
    np.deg2rad(10.0),
    0.0,
    0.0,
)

orbit_offset = build_orbit(yaw_10_deg_qsw)
q_offset = orbit_offset.get_attitude(t_query)

print("QSW + 10 deg yaw first q:", q_offset[0])
print("QSW + 10 deg yaw differs from default vvlh:", bool(not np.allclose(q_offset, q_default)))

QSW + 10 deg yaw first q: [-0.68087653  0.31749794 -0.33000083 -0.57157821]
QSW + 10 deg yaw differs from default vvlh: True


In [8]:
def pitched_vvlh_callable(inertial_frame, iers, simple_eop):
    del iers, simple_eop
    return LofOffset(
        inertial_frame,
        LOFType.VVLH,
        RotationOrder.ZYX,
        0.0,
        np.deg2rad(-5.0),
        0.0,
    )


orbit_offset.set_attitude_law(pitched_vvlh_callable)
q_callable_offset = orbit_offset.get_attitude(t_query)

orbit_offset.set_attitude_law({
    "provider": LofOffset(
        orbit_offset.get_native_frame(),
        LOFType.TNW,
        RotationOrder.ZYX,
        0.0,
        0.0,
        np.deg2rad(15.0),
    )
})
q_mapping_offset = orbit_offset.get_attitude(t_query)

print("VVLH - 5 deg pitch via callable first q:", q_callable_offset[0])
print("TNW + 15 deg roll via provider mapping first q:", q_mapping_offset[0])

VVLH - 5 deg pitch via callable first q: [ 0.06295372 -0.93587869 -0.0456952   0.3436275 ]
TNW + 15 deg roll via provider mapping first q: [ 0.34424589 -0.59409937  0.70376503  0.18236088]


## Practical guidance

- Use `vvlh` as the default for a general Earth-observing satellite.
- Use `lvlh` or `qsw` when you want the STK/Vallado-style LVLH axis convention explicitly.
- Use `nadir` when you want Orekit's nadir-pointing provider tied to the WGS84 Earth shape rather than a pure local-orbital-frame offset.
- Use strings or `LOFType` for built-in aligned LOFs.
- Use `LofOffset` when you need a fixed angular bias from a LOF, such as a yaw-steered or rolled sensor frame.